# 03 - Exercise 2 Production Solution

This notebook is the fast, production-style path for Exercise 2.

Why it exists:

- the walkthrough notebook is optimized for understanding
- this notebook is optimized for running the solution quickly and exporting the final files

Final files produced by this notebook:

- `exercise_2_answer.tsv`
- `outputs/exercise_2_answer.tsv`
- `outputs/exercise_2_answer.tsv`
- `outputs/top_10_songs_in_top_50_sessions.tsv`
- `outputs/top_50_longest_sessions.tsv`
- `outputs/analysis_summary.md`


## Functions used in this notebook

### `create_spark_session(...)`
- **Description:** Starts the Spark session used for the final export notebook.
- **Input:** Notebook app name and optional Spark settings.
- **Output:** Configured `SparkSession`.
- **Why this operation is selected for efficiency:** It centralizes engine tuning so the production notebook stays short and repeatable.

### `load_lastfm_events(...)`
- **Description:** Loads the normalized analytical dataset.
- **Input:** Spark session and project root.
- **Output:** Spark DataFrame of play events.
- **Why this operation is selected for efficiency:** Reusing staged Parquet keeps reruns fast.

### `summarize_sessions_from_events(...)`
- **Description:** Produces one row per session directly from events with Spark session windows.
- **Input:** Event DataFrame and session gap rule.
- **Output:** Session summary DataFrame.
- **Why this operation is selected for efficiency:** This is the core scalable step for Exercise 2 because it aggregates at session level without storing a huge intermediate row-level table.

### `top_longest_sessions_by_track_count(...)`
- **Description:** Selects the top 50 longest sessions by track count.
- **Input:** Session summary DataFrame and limit size.
- **Output:** Top 50 session table.
- **Why this operation is selected for efficiency:** It reduces the data before the final song aggregation.

### `top_songs_from_longest_sessions(...)`
- **Description:** Computes the final ranked song table inside the selected sessions.
- **Input:** Event DataFrame, session summary DataFrame, number of sessions, and number of songs.
- **Output:** Top 10 song DataFrame.
- **Why this operation is selected for efficiency:** Only the 50 selected sessions are joined back to the fact table, which makes the final aggregation compact and scalable.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('C:/Users/GonzaloFigueroa/Documents/Private/BME/coding challenge')

In [2]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

from src.spark_utils import create_spark_session
from src.sessionization import (
    ensure_lastfm_dataset_available,
    load_lastfm_events,
    resolve_lastfm_input_path,
    stage_lastfm_events_to_parquet,
    summarize_sessions_from_events,
    top_longest_sessions_by_track_count,
    top_songs_from_longest_sessions,
)


In [3]:
outputs_dir = PROJECT_ROOT / 'outputs'
outputs_dir.mkdir(exist_ok=True)

spark = create_spark_session(app_name='03-exercise-2-production-solution')
spark


In [4]:
raw_input_path = ensure_lastfm_dataset_available(PROJECT_ROOT)
parquet_path = stage_lastfm_events_to_parquet(spark, PROJECT_ROOT)
print(f'Raw dataset path: {raw_input_path}')
print(f'Prepared Parquet path: {parquet_path}')


Raw dataset path: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\data\raw\lastfm-dataset-1K\userid-timestamp-artid-artname-traid-traname.tsv
Prepared Parquet path: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\data\processed\lastfm_events_parquet


In [5]:
events_df = load_lastfm_events(spark, PROJECT_ROOT)
session_summary_df = summarize_sessions_from_events(events_df, gap_minutes=20).persist(StorageLevel.DISK_ONLY)
session_summary_df.count()


1041883

In [6]:
dataset_stats = events_df.agg(
    F.count('*').alias('total_rows'),
    F.countDistinct('user_id').alias('distinct_users'),
).first().asDict()

dataset_stats


{'total_rows': 19150867, 'distinct_users': 992}

In [7]:
top_50_sessions_df = top_longest_sessions_by_track_count(
    session_summary_df=session_summary_df,
    top_session_count=50,
)

top_10_songs_df = top_songs_from_longest_sessions(
    events_df,
    session_summary_df=session_summary_df,
    top_session_count=50,
    top_song_count=10,
)


In [8]:
top_50_sessions_pd = top_50_sessions_df.toPandas()
top_10_songs_pd = top_10_songs_df.toPandas()

top_10_songs_pd


,artist_name,track_name,play_count,session_count,distinct_track_ids
0,Cake,Jolene,1214,12,1
1,The Knife,Heartbeats,868,2,1
2,Jeff Buckley & Gary Lucas,How Long Will It Take,726,2,1
3,Broken Social Scene,Anthems For A Seventeen Year Old Girl,659,6,1
4,Elliott Smith,St. Ides Heaven,646,6,1
5,The Killers,Bonus Track,634,12,0
6,2Pac,Starin' Through My Rear View,617,12,1
7,The Rolling Stones,Beast Of Burden,613,3,1
8,Everclear,The Swing,604,15,1
9,Kanye West,See You In My Nightmares,536,3,1


In [9]:
top_50_sessions_pd


,user_id,track_count,session_start,session_end,session_number,session_id,session_duration_seconds,session_duration_minutes
0,user_000949,5360,2006-02-12 17:49:31,2006-02-27 11:29:37,151,user_000949-00000151,1273206,21220.100000
1,user_000544,5350,2007-02-12 13:03:52,2007-02-23 00:51:08,75,user_000544-00000075,906436,15107.266667
2,user_000949,4956,2005-12-09 08:26:38,2005-12-18 04:40:04,139,user_000949-00000139,764006,12733.433333
3,user_000949,4705,2007-05-01 02:41:15,2007-05-14 00:05:52,559,user_000949-00000559,1113877,18564.616667
4,user_000997,4357,2007-04-26 00:36:02,2007-05-10 17:55:03,18,user_000997-00000018,1271941,21199.016667
5,user_000544,3809,2007-01-14 04:15:54,2007-01-20 14:31:38,56,user_000544-00000056,555344,9255.733333
6,user_000544,3651,2007-01-06 01:07:04,2007-01-13 13:57:45,55,user_000544-00000055,651041,10850.683333
7,user_000949,3077,2005-11-11 03:30:37,2005-11-18 22:50:07,125,user_000949-00000125,674370,11239.500000
8,user_000262,2862,2008-09-24 08:00:00,2008-09-24 19:59:45,1120,user_000262-00001120,43185,719.750000
9,user_000949,2834,2006-03-18 23:04:14,2006-03-26 18:13:45,189,user_000949-00000189,673771,11229.516667


In [10]:
exercise_2_answer_tsv_path = PROJECT_ROOT / 'exercise_2_answer.tsv'
exercise_2_answer_tsv_export_path = outputs_dir / 'exercise_2_answer.tsv'
outputs_exercise_2_answer_tsv_path = outputs_dir / 'exercise_2_answer.tsv'
top_songs_tsv_path = outputs_dir / 'top_10_songs_in_top_50_sessions.tsv'
top_sessions_tsv_path = outputs_dir / 'top_50_longest_sessions.tsv'
summary_md_path = outputs_dir / 'analysis_summary.md'

top_50_sessions_pd.to_csv(top_sessions_tsv_path, index=False, sep='	')
top_10_songs_pd.to_csv(top_songs_tsv_path, index=False, sep='	')
top_10_songs_pd.to_csv(exercise_2_answer_tsv_export_path, index=False, sep='	')
top_10_songs_pd.to_csv(exercise_2_answer_tsv_path, index=False, sep='	')
top_10_songs_pd.to_csv(outputs_exercise_2_answer_tsv_path, index=False, sep='	')

summary_lines = [
    '# Coding Challenge - Exercise 2 Summary',
    '',
    '## Dataset',
    '',
    f"- Raw input file: `{raw_input_path}`",
    f"- Prepared Parquet dataset: `{parquet_path}`",
    f"- Total valid rows loaded: `{dataset_stats['total_rows']}`",
    f"- Distinct users: `{dataset_stats['distinct_users']}`",
    '',
    '## Question',
    '',
    'What are the top 10 songs played in the top 50 longest sessions by track count?',
    '',
    '## Answer',
    '',
]
for idx, row in top_10_songs_pd.iterrows():
    summary_lines.append(f"{idx + 1}. {row['artist_name']} - {row['track_name']} (plays={row['play_count']}, sessions={row['session_count']})")
summary_lines.extend([
    '',
    '## Output files',
    '',
    f"- `{exercise_2_answer_tsv_path.name}`",
    f"- `{exercise_2_answer_tsv_export_path.name}`",
    f"- `outputs/{outputs_exercise_2_answer_tsv_path.name}`",
    f"- `{top_songs_tsv_path.name}`",
    f"- `{top_sessions_tsv_path.name}`",
])
summary_md_path.write_text('\n'.join(summary_lines), encoding='utf-8')

print(f'Saved: {exercise_2_answer_tsv_path}')
print(f'Saved: {exercise_2_answer_tsv_export_path}')
print(f'Saved: {outputs_exercise_2_answer_tsv_path}')
print(f'Saved: {top_songs_tsv_path}')
print(f'Saved: {top_sessions_tsv_path}')
print(f'Saved: {summary_md_path}')


Saved: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\exercise_2_answer.tsv
Saved: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\outputs\exercise_2_answer.tsv
Saved: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\outputs\exercise_2_answer.tsv
Saved: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\outputs\top_10_songs_in_top_50_sessions.tsv
Saved: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\outputs\top_50_longest_sessions.tsv
Saved: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\outputs\analysis_summary.md


In [11]:
for idx, row in top_10_songs_pd.iterrows():
    print(f"{idx + 1}. {row['artist_name']} - {row['track_name']} (plays={row['play_count']}, sessions={row['session_count']})")


1. Cake - Jolene (plays=1214, sessions=12)
2. The Knife - Heartbeats (plays=868, sessions=2)
3. Jeff Buckley & Gary Lucas - How Long Will It Take (plays=726, sessions=2)
4. Broken Social Scene - Anthems For A Seventeen Year Old Girl (plays=659, sessions=6)
5. Elliott Smith - St. Ides Heaven (plays=646, sessions=6)
6. The Killers - Bonus Track (plays=634, sessions=12)
7. 2Pac - Starin' Through My Rear View (plays=617, sessions=12)
8. The Rolling Stones - Beast Of Burden (plays=613, sessions=3)
9. Everclear - The Swing (plays=604, sessions=15)
10. Kanye West - See You In My Nightmares (plays=536, sessions=3)


In [ ]:
spark.stop()
